In [2]:
import torch
import json
print(f"CUDA 可用性: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("❌ 警告：当前环境没有 GPU，无法进行归因分析！")

CUDA 可用性: True


In [1]:
# environment

import torch
import json
import chromadb
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_PATH = "/space_mounts/pars/RAG_Project"
DB_PATH = "./chroma_db"
METADATA_PATH = f"{BASE_PATH}/target_metadata.json"

# load model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager", 
    torch_dtype=torch.float16
)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [2]:
chroma_client = chromadb.PersistentClient(path=DB_PATH)
collections = {
    "targeted": chroma_client.get_collection(name="targeted_poison_set")
}

with open(METADATA_PATH, "r") as f:
    target_questions = json.load(f)

In [5]:
from tqdm import tqdm
import gc

def run_interpretability_batch(target_questions, model, tokenizer, collections):
    interpret_results = []
    
    print("starting...")
    for item in tqdm(target_questions):
        q_id = item['id']
        q_text = item['question']
        
        try:
            t_doc = collections['targeted'].get(ids=[f"targeted_poison_{q_id}"])['documents'][0]
   
            input_text = f"Context: {t_doc}\nQuestion: {q_text}\nAnswer:"
            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
            input_ids = inputs.input_ids
    
            embeddings = model.get_input_embeddings()(input_ids).detach().requires_grad_(True)
        
            outputs = model(inputs_embeds=embeddings, output_attentions=True)
            logits = outputs.logits[:, -1, :]
            
            target_id = tokenizer.encode(item['answer'], add_special_tokens=False)[0]
            target_logit = logits[0, target_id]
      
            model.zero_grad()
            target_logit.backward(retain_graph=True)

            saliency = embeddings.grad.data.abs().sum(dim=-1).squeeze(0)
            saliency = (saliency / (saliency.max() + 1e-8)).tolist() 
          
            last_layer_attn = outputs.attentions[-1] 
            avg_attn = last_layer_attn.mean(dim=1).squeeze(0) # [seq_len, seq_len]
            final_token_attn = avg_attn[-1, :].tolist()
            

            tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
            tokens = [t.replace('Ġ', ' ').replace(' ', ' ') for t in tokens]
            
            interpret_results.append({
                "id": q_id,
                "tokens": tokens,
                "attribution": saliency,
                "attention": final_token_attn,
                "target_predicted": item['answer']
            })

            del outputs, logits, embeddings, saliency # 删除占用巨大的中间变量
            torch.cuda.empty_cache() # 强制清理显存碎片
            gc.collect()
            
        except Exception as e:
            print(f"⚠️ skip {q_id}: {e}")
            torch.cuda.empty_cache()
            
    return interpret_results

interpretability_data = run_interpretability_batch(target_questions, model, tokenizer, collections)

# 保存到本地 files，方便后续 Streamlit 读取
INTERPRET_SAVE_PATH = "./interpretability_results.json"
with open(INTERPRET_SAVE_PATH, "w", encoding='utf-8') as f:
    json.dump(interpretability_data, f, ensure_ascii=False, indent=2)

print(f"✅ saved to: {INTERPRET_SAVE_PATH}")

starting...


100%|██████████| 50/50 [00:37<00:00,  1.35it/s]

✅ saved to: ./interpretability_results.json


In [6]:
import json

# 读取刚生成的数据
with open("./interpretability_results.json", "r") as f:
    interp_data = json.load(f)

# 看看第一个案例
sample = interp_data[0]
print(f"分析目标 (假答案): {sample['target_predicted']}")
print("-" * 50)
print(f"{'Token':<15} | {'归因分数 (Attribution)':<20} | {'注意力 (Attention)':<15}")
print("-" * 50)

# 只打印分数较高的前 10 个词，看看是谁在“作妖”
top_tokens = sorted(zip(sample['tokens'], sample['attribution'], sample['attention']), 
                    key=lambda x: x[1], reverse=True)[:15]

for t, attr, attn in top_tokens:
    print(f"{t:<15} | {attr:<20.4f} | {attn:<15.4f}")

分析目标 (假答案): Arthur's Magazine
--------------------------------------------------
Token           | 归因分数 (Attribution)   | 注意力 (Attention)
--------------------------------------------------
Context         | nan                  | 0.1915         
:               | nan                  | 0.0042         
 [              | nan                  | 0.0070         
factor          | nan                  | 0.0074         
 factor         | nan                  | 0.0032         
]               | nan                  | 0.0108         
 Update         | nan                  | 0.0072         
 (              | nan                  | 0.0053         
May             | nan                  | 0.0024         
                | nan                  | 0.0009         
2               | nan                  | 0.0003         
0               | nan                  | 0.0001         
2               | nan                  | 0.0035         
6               | nan                  | 0.0008         
):           